<a href="https://colab.research.google.com/github/cdhaskett/GB885-Final-Project-Haskett-C/blob/main/GB885_Final_Project_Haskett_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

As a sales analyst for "RUSH," you need to review raw sales data of three tables that include products, retailers and sales. You will need to perform EDA, answer VP questions and identify trends and insights. After that, you need to create a GitHub repository and a recorded presentation. All of these items will be available on the repository.

In [85]:
#Let's import the needed libraries:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

In [86]:
# Load the raw data directly from the GitHub repository
base_url = "https://raw.githubusercontent.com/cdhaskett/GB885-Final-Project-Haskett-C/main/data/"

In [87]:
# Create separate dataframes for each table
df_products = pd.read_csv(base_url + "TABLE_PRODUCTS_885.csv")
df_retailers = pd.read_csv(base_url + "TABLE_RETAILER_885.csv")
df_sales = pd.read_csv(base_url + "TABLE_SALES_885.csv")

In [88]:
#Let's take a look at all three dataframe headers:
df_products.head()

,PRODUCT_ID|PRODUCT_NAME
0,020|Men's Street Footwear
1,030|Men's Athletic Footwear
2,120|Women's Street Footwear
3,130|Women's Athletic Footwear
4,040|Men's Apparel


In [89]:
#First thing I noticed is that the products table is pipe delimited. Let's make sure we split them correctly. Since it's all one conglomerate we also need to convert product_id to a numeric type.

In [90]:
# Split 'PRODUCT_ID|PRODUCT_NAME' into two columns in df_products
df_products[['PRODUCT_ID', 'PRODUCT_NAME']] = df_products['PRODUCT_ID|PRODUCT_NAME'].str.split('|', expand=True)

# Convert PRODUCT_ID to numeric type
df_products['PRODUCT_ID'] = pd.to_numeric(df_products['PRODUCT_ID'])

# Drop the original combined column
df_products = df_products.drop(columns=['PRODUCT_ID|PRODUCT_NAME'])

print("df_products after cleaning:")
display(df_products.head())
print("\nUpdated df_products info:")
df_products.info()

df_products after cleaning:


,PRODUCT_ID,PRODUCT_NAME
0,20,Men's Street Footwear
1,30,Men's Athletic Footwear
2,120,Women's Street Footwear
3,130,Women's Athletic Footwear
4,40,Men's Apparel



Updated df_products info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   PRODUCT_ID    6 non-null      int64 
 1   PRODUCT_NAME  6 non-null      object
dtypes: int64(1), object(1)
memory usage: 228.0+ bytes


In [91]:
#Now let's look at df_sales
df_sales.head()
df_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9648 entries, 0 to 9647
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDER_ID          9648 non-null   int64  
 1   RETAILER_ID       9648 non-null   object 
 2   INVOICE_DATE      9648 non-null   object 
 3   MONTH             9648 non-null   int64  
 4   DAY               9648 non-null   int64  
 5   YEAR              9648 non-null   int64  
 6   PRODUCT_ID        9648 non-null   int64  
 7   PRICE_PER_UNIT    9646 non-null   float64
 8   UNITS_SOLD        9648 non-null   object 
 9   OPERATING_MARGIN  9648 non-null   float64
 10  SALES_METHOD      9648 non-null   object 
dtypes: float64(2), int64(5), object(4)
memory usage: 829.3+ KB


In [92]:
#Right away I see some type mismatches--let's address those.

# Convert 'INVOICE_DATE' to datetime objects in df_sales
df_sales['INVOICE_DATE'] = pd.to_datetime(df_sales['INVOICE_DATE'])

# Convert 'UNITS_SOLD' to numeric type, coercing errors to NaN
df_sales['UNITS_SOLD'] = pd.to_numeric(df_sales['UNITS_SOLD'], errors='coerce')

print("df_sales types after conversion:")
df_sales.info()

df_sales types after conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9648 entries, 0 to 9647
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ORDER_ID          9648 non-null   int64         
 1   RETAILER_ID       9648 non-null   object        
 2   INVOICE_DATE      9648 non-null   datetime64[ns]
 3   MONTH             9648 non-null   int64         
 4   DAY               9648 non-null   int64         
 5   YEAR              9648 non-null   int64         
 6   PRODUCT_ID        9648 non-null   int64         
 7   PRICE_PER_UNIT    9646 non-null   float64       
 8   UNITS_SOLD        9646 non-null   float64       
 9   OPERATING_MARGIN  9648 non-null   float64       
 10  SALES_METHOD      9648 non-null   object        
dtypes: datetime64[ns](1), float64(3), int64(5), object(2)
memory usage: 829.3+ KB


In [93]:
# Check for null values in df_sales
print("Null values in df_sales:")
display(df_sales.isnull().sum())

# Check for duplicate rows in df_sales
print("\nNumber of duplicate rows in df_sales:")
display(df_sales.duplicated().sum())

Null values in df_sales:


,0
ORDER_ID,0
RETAILER_ID,0
INVOICE_DATE,0
MONTH,0
DAY,0
YEAR,0
PRODUCT_ID,0
PRICE_PER_UNIT,2
UNITS_SOLD,2
OPERATING_MARGIN,0



Number of duplicate rows in df_sales:


np.int64(0)

In [94]:
#Since we see there are 2 in units_sold and 2 in price_per_unit--let's bring those up. If rows contain valuable information--we will keep them. It looks like there are only a few missing items from those four rows.
missing_values_rows = df_sales[df_sales['PRICE_PER_UNIT'].isnull() | df_sales['UNITS_SOLD'].isnull()]
display(missing_values_rows)

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD
98,591,A00NVEBU,2020-04-02,4,2,2020,20,NaN,525.0,0.35,In-store
99,597,A00NVEBU,2020-04-08,4,8,2020,20,NaN,525.0,0.50,In-store
1012,6064,S00SALBI,2021-05-27,5,27,2021,20,51.0,NaN,0.45,Online
1439,8626,W00MIODE,2021-12-10,12,10,2021,20,29.0,NaN,0.46,Outlet


In [95]:
# Now let's look at the retailer dataframe
df_retailers.head()
df_retailers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   RETAILER_ID  110 non-null    object
 1   RETAILER     110 non-null    object
 2   REGION       110 non-null    object
 3   STATE        110 non-null    object
 4   CITY         110 non-null    object
dtypes: object(5)
memory usage: 4.4+ KB


In [96]:
# Check for missing values
df_retailers.isnull().sum()


,0
RETAILER_ID,0
RETAILER,0
REGION,0
STATE,0
CITY,0


In [97]:
# Check for completely duplicated rows
df_retailers.duplicated().sum()

np.int64(0)

In [98]:
# Check whether RETAILER_ID is actually unique
df_retailers['RETAILER_ID'].duplicated().sum()

np.int64(4)

In [99]:
# Display all records with duplicate retailer IDs
duplicate_retailers = df_retailers[
    df_retailers['RETAILER_ID'].duplicated(keep=False)
].sort_values('RETAILER_ID')

display(duplicate_retailers)

,RETAILER_ID,RETAILER,REGION,STATE,CITY
63,S00NNENE,Sports Direct,Northeast,New Jersey,Newark
64,S00NNENE,Sports Direct,Northeast,New York,New York
81,W00SARLI,Walmart,South,Arkansas,Little Rock
97,W00SARLI,West Gear,South,Arkansas,Little Rock
84,W00SFLOR,Walmart,Southeast,Florida,Orlando
102,W00SFLOR,West Gear,Southeast,Florida,Orlando
83,W00STEHO,Walmart,South,Texas,Houston
100,W00STEHO,West Gear,South,Texas,Houston


In [100]:
# Compare retailer ID patterns for the affected retailers
df_retailers[
    df_retailers['RETAILER'].isin(['Sports Direct', 'Walmart', 'West Gear'])
][['RETAILER_ID', 'RETAILER', 'REGION', 'STATE', 'CITY']].sort_values(
    ['RETAILER', 'STATE', 'CITY']
)

,RETAILER_ID,RETAILER,REGION,STATE,CITY
66,S00SALBI,Sports Direct,South,Alabama,Birmingham
60,S00NCOHA,Sports Direct,Northeast,Connecticut,Hartford
71,S00SFLMI,Sports Direct,Southeast,Florida,Miami
72,S00SFLOR,Sports Direct,Southeast,Florida,Orlando
73,S00SGEAT,Sports Direct,Southeast,Georgia,Atlanta
77,S00WHAHO,Sports Direct,West,Hawaii,Honolulu
78,S00WIDBO,Sports Direct,West,Idaho,Boise
54,S00MILCH,Sports Direct,Midwest,Illinois,Chicago
61,S00NMABA,Sports Direct,Northeast,Maryland,Baltimore
62,S00NMABO,Sports Direct,Northeast,Massachusetts,Boston


The `RETAILER_ID` field was intended to uniquely identify each retailer-location combination. However, the ID appears to be created from abbreviated retailer and location information. In several cases, those abbreviations are not specific enough to remain unique.

For example, Walmart and West Gear in the same city can receive the same `RETAILER_ID`, and the Sports Direct locations in Newark, New Jersey and New York, New York also share an ID.

This indicates that the ID-generation method created collisions in the raw data. Because `RETAILER_ID` is used as the key between the retailer and sales tables, these collisions must be resolved before merging the data.

In [101]:
# Preserve the original ID and create a cleaned version
df_retailers['RETAILER_ID_CLEAN'] = df_retailers['RETAILER_ID']

In [102]:
# Create improved IDs only for the duplicate retailer records

state_codes = {
    'Arkansas': 'AR',
    'Florida': 'FL',
    'Texas': 'TX',
    'New Jersey': 'NJ',
    'New York': 'NY'
}

region_codes = {
    'South': 'S',
    'Southeast': 'S',
    'Northeast': 'N'
}

duplicate_mask = df_retailers['RETAILER_ID'].duplicated(keep=False)

df_retailers.loc[duplicate_mask, 'RETAILER_ID_CLEAN'] = (
    df_retailers.loc[duplicate_mask, 'RETAILER']
        .str.replace(' ', '', regex=False)
        .str[:2]
        .str.upper()
    +
    df_retailers.loc[duplicate_mask, 'REGION'].map(region_codes)
    +
    df_retailers.loc[duplicate_mask, 'STATE'].map(state_codes)
    +
    df_retailers.loc[duplicate_mask, 'CITY']
        .str.replace(' ', '', regex=False)
        .str[:3]
        .str.upper()
)

In [103]:
#Let's check out the changed ids only.
df_retailers.loc[
    duplicate_mask,
    ['RETAILER_ID', 'RETAILER_ID_CLEAN', 'RETAILER', 'STATE', 'CITY']
]

,RETAILER_ID,RETAILER_ID_CLEAN,RETAILER,STATE,CITY
63,S00NNENE,SPNNJNEW,Sports Direct,New Jersey,Newark
64,S00NNENE,SPNNYNEW,Sports Direct,New York,New York
81,W00SARLI,WASARLIT,Walmart,Arkansas,Little Rock
83,W00STEHO,WASTXHOU,Walmart,Texas,Houston
84,W00SFLOR,WASFLORL,Walmart,Florida,Orlando
97,W00SARLI,WESARLIT,West Gear,Arkansas,Little Rock
100,W00STEHO,WESTXHOU,West Gear,Texas,Houston
102,W00SFLOR,WESFLORL,West Gear,Florida,Orlando


In [104]:
# Make sure the cleaned IDs are now unique
df_retailers['RETAILER_ID_CLEAN'].duplicated().sum()

np.int64(0)

To resolve the duplicate retailer IDs, I created a cleaned version of the ID only for the affected records. The revised IDs use additional retailer and location information while preserving the original ID structure as much as possible. After making these corrections, `RETAILER_ID_CLEAN` contained no duplicate values.

In [105]:
# Preserve the original retailer ID in sales
df_sales['RETAILER_ID_CLEAN'] = df_sales['RETAILER_ID']

### Correcting Retailer IDs in the Sales Table

The duplicate Walmart and West Gear IDs were investigated using the order sequence in the sales table. The affected orders appeared in distinct retailer blocks surrounded by known Walmart or West Gear records.

I used these patterns to assign the affected sales to the appropriate retailer while preserving the original `RETAILER_ID`. The Sports Direct records could not be reliably separated between Newark, New Jersey and New York, New York, so those records were left unresolved rather than assigned arbitrarily.

In [106]:
# Fix Walmart records
df_sales.loc[
    (df_sales['RETAILER_ID'] == 'W00SFLOR') &
    (df_sales['ORDER_ID'].between(6650, 6975)),
    'RETAILER_ID_CLEAN'
] = 'WASFLORL'

df_sales.loc[
    (df_sales['RETAILER_ID'] == 'W00SARLI') &
    (df_sales['ORDER_ID'].between(6799, 7124)),
    'RETAILER_ID_CLEAN'
] = 'WASARLIT'

df_sales.loc[
    (df_sales['RETAILER_ID'] == 'W00STEHO') &
    (df_sales['ORDER_ID'].between(6909, 7274)),
    'RETAILER_ID_CLEAN'
] = 'WASTXHOU'

In [107]:
# Fix West Gear records
df_sales.loc[
    (df_sales['RETAILER_ID'] == 'W00SFLOR') &
    (df_sales['ORDER_ID'] >= 7485),
    'RETAILER_ID_CLEAN'
] = 'WESFLORL'

df_sales.loc[
    (df_sales['RETAILER_ID'] == 'W00SARLI') &
    (df_sales['ORDER_ID'] >= 8439),
    'RETAILER_ID_CLEAN'
] = 'WESARLIT'

df_sales.loc[
    (df_sales['RETAILER_ID'] == 'W00STEHO') &
    (df_sales['ORDER_ID'] >= 7473),
    'RETAILER_ID_CLEAN'
] = 'WESTXHOU'

In [108]:
# Check remaining sales with the original duplicate IDs
df_sales[
    df_sales['RETAILER_ID_CLEAN'].isin(
        ['W00SFLOR', 'W00SARLI', 'W00STEHO', 'S00NNENE']
    )
]['RETAILER_ID_CLEAN'].value_counts()

,count
RETAILER_ID_CLEAN,
S00NNENE,56


We have fixed the majority of the issues and are now left with the Sports Direct that are in the New York & New Jersey area.

In [109]:
#Since we struggled with the cleaning part--let's do the merging in two steps to make sure we can validate each.

# Merge sales with product information
df_merged = df_sales.merge(
    df_products,
    on='PRODUCT_ID',
    how='left',
    validate='m:1'
)

print("Rows after product merge:", len(df_merged))

Rows after product merge: 9648


In [110]:
# Merge retailer information using the cleaned retailer ID
df_merged = df_merged.merge(
    df_retailers,
    on='RETAILER_ID_CLEAN',
    how='left',
    validate='m:1',
    indicator='retailer_match'
)

print("Rows after retailer merge:", len(df_merged))

Rows after retailer merge: 9648


In [111]:
#Let's check the values
df_merged['retailer_match'].value_counts()

,count
retailer_match,
both,9591
left_only,57
right_only,0


In [112]:
#Let's count the values
df_merged[
    df_merged['retailer_match'] == 'left_only'
]['RETAILER_ID_CLEAN'].value_counts()

,count
RETAILER_ID_CLEAN,
S00NNENE,56
999999999,1


In [113]:
# Investigate the unmatched retailer ID
df_merged[
    df_merged['RETAILER_ID_CLEAN'] == '999999999'
][[
    'ORDER_ID',
    'RETAILER_ID_x',
    'INVOICE_DATE',
    'PRODUCT_ID',
    'PRICE_PER_UNIT',
    'UNITS_SOLD',
    'SALES_METHOD'
]]

,ORDER_ID,RETAILER_ID_x,INVOICE_DATE,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,SALES_METHOD
1446,8668,999999999,2021-07-23,20,60.0,298.0,Outlet


One sales record contained the retailer ID `999999999`, which did not match any record in the retailer table. Because the remaining sales information was valid, I kept the transaction for overall sales and product analysis but treated its retailer and location as unknown.

In [114]:
#Clean up retailer ID columns after the merge
df_merged = df_merged.rename(columns={
    'RETAILER_ID_x': 'RETAILER_ID_ORIGINAL',
    'RETAILER_ID_y': 'RETAILER_ID_RETAILER_TABLE'
})

In [115]:
#Calculate sales dollars for each order
df_merged['SALES_DOLLARS'] = (
    df_merged['PRICE_PER_UNIT'] *
    df_merged['UNITS_SOLD']
)

In [116]:
# Check how many orders cannot have sales dollars calculated
df_merged['SALES_DOLLARS'].isnull().sum()

np.int64(4)

Four sales records could not have total sales dollars calculated because either `PRICE_PER_UNIT` or `UNITS_SOLD` was missing in the raw data. These records were kept in the dataset but will be excluded from calculations that require `SALES_DOLLARS`.

In [117]:
#Lot's of work--let's check it out.
df_merged[
    ['ORDER_ID', 'PRODUCT_NAME', 'PRICE_PER_UNIT',
     'UNITS_SOLD', 'SALES_DOLLARS']
].head()

,ORDER_ID,PRODUCT_NAME,PRICE_PER_UNIT,UNITS_SOLD,SALES_DOLLARS
0,1,Men's Street Footwear,50.0,1200.0,60000.0
1,7,Men's Street Footwear,50.0,1250.0,62500.0
2,13,Men's Street Footwear,50.0,1220.0,61000.0
3,19,Men's Street Footwear,50.0,1200.0,60000.0
4,25,Men's Street Footwear,60.0,1220.0,73200.0


In [118]:
# Final primary key checks
print("Duplicate Order IDs:",
      df_sales['ORDER_ID'].duplicated().sum())

print("Duplicate Product IDs:",
      df_products['PRODUCT_ID'].duplicated().sum())

print("Duplicate Clean Retailer IDs:",
      df_retailers['RETAILER_ID_CLEAN'].duplicated().sum())

Duplicate Order IDs: 0
Duplicate Product IDs: 0
Duplicate Clean Retailer IDs: 0


In [119]:
# Confirm the merge did not add or remove sales records
print("Original sales rows:", len(df_sales))
print("Merged rows:", len(df_merged))
print("Unique orders in merged data:", df_merged['ORDER_ID'].nunique())

Original sales rows: 9648
Merged rows: 9648
Unique orders in merged data: 9648


### Data Cleaning Summary

After reviewing the raw tables, I corrected data type issues, investigated missing values, identified retailer ID collisions, created cleaned retailer IDs for the affected records, and validated the table relationships before merging.

The final merged dataset retained the same number of sales records as the original sales table, confirming that the merge did not duplicate or remove orders. The cleaned data is now ready for sales analysis.

In [120]:
#Let's first find out what category had teh highest sales in 2021 and how much it sold.

#Filter the data to 2021
df_2021 = df_merged[df_merged['YEAR'] == 2021]

#Calculate sales by product category
product_sales_2021 = (
    df_2021.groupby('PRODUCT_NAME')['SALES_DOLLARS']
    .sum()
    .sort_values(ascending=False)
)

product_sales_2021

,SALES_DOLLARS
PRODUCT_NAME,
Men's Street Footwear,74648880.0
Women's Apparel,19178278.0
Men's Athletic Footwear,16339593.0
Women's Street Footwear,13535783.0
Men's Apparel,13025293.0
Women's Athletic Footwear,11160629.0


In [121]:
# Display the highest-selling product category
top_product = product_sales_2021.idxmax()
top_product_sales = product_sales_2021.max()

print("Top Product Category:", top_product)
print(f"2021 Sales: ${top_product_sales:,.2f}")

Top Product Category: Men's Street Footwear
2021 Sales: $74,648,880.00


In [122]:
#What states had teh highest sales of women's products in 2021 and how much was sold?
# Filter to women's products sold in 2021
women_2021 = df_2021[
    df_2021['PRODUCT_NAME'].str.contains("Women's", na=False)
]

# Calculate women's sales by state
women_state_sales = (
    women_2021.groupby('STATE')['SALES_DOLLARS']
    .sum()
    .sort_values(ascending=False)
)

women_state_sales.head(10)

,SALES_DOLLARS
STATE,
Maine,2176301.0
Delaware,2023575.0
New Hampshire,1916400.0
Arizona,1798900.0
Missouri,1771992.0
Illinois,1743277.0
Virginia,1719886.0
Nebraska,1712680.0
Connecticut,1600156.0


In [123]:
#Display the top state for women's product sales
top_women_state = women_state_sales.idxmax()
top_women_sales = women_state_sales.max()

print("Top State:", top_women_state)
print(f"Women's Product Sales: ${top_women_sales:,.2f}")

Top State: Maine
Women's Product Sales: $2,176,301.00


In [124]:
#What state had the highest sales of men's products in 2021 and how much was sold?

#Filter to men's products sold in 2021
men_2021 = df_2021[
    df_2021['PRODUCT_NAME'].str.contains("Men's", na=False)
]

#Calculate men's sales by state
men_state_sales = (
    men_2021.groupby('STATE')['SALES_DOLLARS']
    .sum()
    .sort_values(ascending=False)
)

men_state_sales.head(10)

,SALES_DOLLARS
STATE,
New Hampshire,54208080.0
Delaware,2334300.0
Arizona,2261025.0
Maine,2217190.0
Illinois,2093438.0
Missouri,1951530.0
Connecticut,1926568.0
New York,1847069.0
Nebraska,1718726.0


In [125]:
# Display the top state for men's product sales
top_men_state = men_state_sales.idxmax()
top_men_sales = men_state_sales.max()

print("Top State:", top_men_state)
print(f"Men's Product Sales: ${top_men_sales:,.2f}")

Top State: New Hampshire
Men's Product Sales: $54,208,080.00


In [126]:
#What retailer prouchased the most units in 2021 and in 2020?

# Calculate units purchased by retailer in 2021
retailer_units_2021 = (
    df_merged[df_merged['YEAR'] == 2021]
    .groupby('RETAILER')['UNITS_SOLD']
    .sum()
    .sort_values(ascending=False)
)

retailer_units_2021

,UNITS_SOLD
RETAILER,
Foot Locker,1097410.0
West Gear,279762.0
Sports Direct,239899.0
Amazon,205570.0
Kohl's,136950.0
Walmart,48010.0


In [127]:
top_retailer_2021 = retailer_units_2021.idxmax()
top_units_2021 = retailer_units_2021.max()

print("Top Retailer in 2021:", top_retailer_2021)
print(f"Units Purchased: {top_units_2021:,.0f}")

Top Retailer in 2021: Foot Locker
Units Purchased: 1,097,410


In [128]:
# Calculate units purchased by retailer in 2020
retailer_units_2020 = (
    df_merged[df_merged['YEAR'] == 2020]
    .groupby('RETAILER')['UNITS_SOLD']
    .sum()
    .sort_values(ascending=False)
)

retailer_units_2020

,UNITS_SOLD
RETAILER,
Amazon,317930.0
Kohl's,68686.0
West Gear,57334.0
Sports Direct,18399.0


In [129]:
top_retailer_2020 = retailer_units_2020.idxmax()
top_units_2020 = retailer_units_2020.max()

print("Top Retailer in 2020:", top_retailer_2020)
print(f"Units Purchased: {top_units_2020:,.0f}")

Top Retailer in 2020: Amazon
Units Purchased: 317,930


In [130]:
#I am a visual person--so let's do some mapping with plotly.

import plotly.express as px

In [131]:
# Calculate total 2021 sales by state
state_sales_2021 = (
    df_2021.groupby('STATE')['SALES_DOLLARS']
    .sum()
    .reset_index()
)

state_sales_2021.head()

,STATE,SALES_DOLLARS
0,Alabama,1245284.0
1,Alaska,1895800.0
2,Arizona,4059925.0
3,Arkansas,601792.0
4,California,1702158.0


In [132]:
# Create state abbreviation lookup
state_abbreviations = {
    'Alabama':'AL', 'Alaska':'AK', 'Arizona':'AZ', 'Arkansas':'AR',
    'California':'CA', 'Colorado':'CO', 'Connecticut':'CT', 'Delaware':'DE',
    'Florida':'FL', 'Georgia':'GA', 'Hawaii':'HI', 'Idaho':'ID',
    'Illinois':'IL', 'Indiana':'IN', 'Iowa':'IA', 'Kansas':'KS',
    'Kentucky':'KY', 'Louisiana':'LA', 'Maine':'ME', 'Maryland':'MD',
    'Massachusetts':'MA', 'Michigan':'MI', 'Minnesota':'MN', 'Mississippi':'MS',
    'Missouri':'MO', 'Montana':'MT', 'Nebraska':'NE', 'Nevada':'NV',
    'New Hampshire':'NH', 'New Jersey':'NJ', 'New Mexico':'NM', 'New York':'NY',
    'North Carolina':'NC', 'North Dakota':'ND', 'Ohio':'OH', 'Oklahoma':'OK',
    'Oregon':'OR', 'Pennsylvania':'PA', 'Rhode Island':'RI',
    'South Carolina':'SC', 'South Dakota':'SD', 'Tennessee':'TN',
    'Texas':'TX', 'Utah':'UT', 'Vermont':'VT', 'Virginia':'VA',
    'Washington':'WA', 'West Virginia':'WV', 'Wisconsin':'WI', 'Wyoming':'WY'
}

state_sales_2021['STATE_ABBR'] = (
    state_sales_2021['STATE'].map(state_abbreviations)
)

In [133]:
# Create a heat map of 2021 sales by state
fig = px.choropleth(
    state_sales_2021,
    locations='STATE_ABBR',
    locationmode='USA-states',
    color='SALES_DOLLARS',
    scope='usa',
    hover_name='STATE',
    hover_data={'SALES_DOLLARS': ':$,.0f', 'STATE_ABBR': False},
    title='2021 Sales Concentration by State'
)

fig.show()

In [134]:
# Show the top 10 states by 2021 sales
top_states_2021 = (
    state_sales_2021
    .sort_values('SALES_DOLLARS', ascending=False)
    .head(10)
)

top_states_2021[['STATE', 'SALES_DOLLARS']]

,STATE,SALES_DOLLARS
28,New Hampshire,56124480.0
18,Maine,4393491.0
7,Delaware,4357875.0
2,Arizona,4059925.0
12,Illinois,3836715.0
24,Missouri,3723522.0
6,Connecticut,3526724.0
26,Nebraska,3431406.0
45,Virginia,3420809.0
31,New York,3352325.0


In [135]:
# Create a bar chart of the top 10 states
fig = px.bar(
    top_states_2021,
    x='SALES_DOLLARS',
    y='STATE',
    orientation='h',
    title='Top 10 States by Sales in 2021',
    labels={
        'SALES_DOLLARS': 'Sales Dollars',
        'STATE': 'State'
    }
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

In [136]:
# Create a clean dataset for the Streamlit dashboard
app_data = df_merged[
    [
        'ORDER_ID',
        'INVOICE_DATE',
        'YEAR',
        'MONTH',
        'PRODUCT_NAME',
        'PRICE_PER_UNIT',
        'UNITS_SOLD',
        'SALES_DOLLARS',
        'OPERATING_MARGIN',
        'SALES_METHOD',
        'RETAILER',
        'REGION',
        'STATE',
        'CITY'
    ]
].copy()

app_data.to_csv('rush_cleaned_sales.csv', index=False)

print("Dashboard dataset created:", len(app_data), "rows")

Dashboard dataset created: 9648 rows


In [137]:
from google.colab import files

files.download('rush_cleaned_sales.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>